In [1]:
import pandas as pd
import numpy as np
import mlflow
import sys
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports done!")

✅ Imports done!


In [2]:
sys.path.append('../utils')
from snowflake_connector import read_table

# Load from Snowflake
product_affinity = read_table('GOLD_PRODUCT_AFFINITY')
customer_features = read_table('GOLD_CUSTOMER_FEATURES')

# Load model outputs
als_df = pd.read_csv('../data/als_recommendations.csv')
cb_df = pd.read_csv('../data/cb_recommendations.csv')

print("\n✅ All data loaded!")
print(f"Product affinity:    {len(product_affinity)} rows")
print(f"Customer features:   {len(customer_features):,} rows")
print(f"ALS recommendations: {len(als_df):,}")
print(f"CB recommendations:  {len(cb_df):,}")

✅ Loaded GOLD_PRODUCT_AFFINITY: 40 rows
✅ Loaded GOLD_CUSTOMER_FEATURES: 10,000 rows

✅ All data loaded!
Product affinity:    40 rows
Customer features:   10,000 rows
ALS recommendations: 28,920
CB recommendations:  29,966


In [3]:
# Clean column names to lowercase
product_affinity.columns = product_affinity.columns.str.lower()
customer_features.columns = customer_features.columns.str.lower()

# Create affinity lookup dict
# {(segment, product_name): conversion_rate}
affinity_lookup = {}
for _, row in product_affinity.iterrows():
    key = (row['segment'], row['product_name'])
    affinity_lookup[key] = row['conversion_rate']

print("✅ Affinity lookup created!")
print(f"Total segment-product pairs: {len(affinity_lookup)}")
print("\nSample affinity scores:")
for key, rate in list(affinity_lookup.items())[:5]:
    print(f"  {key[0]} + {key[1]}: {rate:.3f}")

✅ Affinity lookup created!
Total segment-product pairs: 40

Sample affinity scores:
  family + checking_account: 1.000
  family + home_loan: 0.527
  family + savings_account: 0.521
  family + fixed_deposit: 0.509
  family + investment_fund: 0.050


In [4]:
# Normalize all scores to 0-1 scale
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Normalize ALS scores
als_df['als_score_norm'] = scaler.fit_transform(als_df[['als_score']])

# Normalize CB scores
cb_df['cb_score_norm'] = scaler.fit_transform(cb_df[['cb_score']])

print("✅ Scores normalized!")
print("\nALS score range:")
print(f"  Before: {als_df['als_score'].min():.4f} - {als_df['als_score'].max():.4f}")
print(f"  After:  {als_df['als_score_norm'].min():.4f} - {als_df['als_score_norm'].max():.4f}")

print("\nCB score range:")
print(f"  Before: {cb_df['cb_score'].min():.4f} - {cb_df['cb_score'].max():.4f}")
print(f"  After:  {cb_df['cb_score_norm'].min():.4f} - {cb_df['cb_score_norm'].max():.4f}")

✅ Scores normalized!

ALS score range:
  Before: 0.0032 - 0.9661
  After:  0.0000 - 1.0000

CB score range:
  Before: 0.1027 - 0.9747
  After:  0.0000 - 1.0000


In [5]:
# Merge ALS and CB recommendations
hybrid_df = pd.merge(
    als_df[['customer_id', 'product_name', 'als_score_norm']],
    cb_df[['customer_id', 'product_name', 'cb_score_norm']],
    on=['customer_id', 'product_name'],
    how='outer'
).fillna(0)

# Add customer segment
customer_segment = customer_features[['customer_id', 'segment']]
hybrid_df = hybrid_df.merge(customer_segment, on='customer_id', how='left')

# Add affinity score
def get_affinity_score(row):
    key = (row['segment'], row['product_name'])
    return affinity_lookup.get(key, 0.0)

hybrid_df['affinity_score'] = hybrid_df.apply(get_affinity_score, axis=1)

# Normalize affinity score
hybrid_df['affinity_score_norm'] = scaler.fit_transform(hybrid_df[['affinity_score']])

# Calculate hybrid score with weights
# ALS: 40%, Content-Based: 40%, Affinity: 20%
ALS_WEIGHT = 0.4
CB_WEIGHT = 0.4
AFFINITY_WEIGHT = 0.2

hybrid_df['hybrid_score'] = (
    ALS_WEIGHT * hybrid_df['als_score_norm'] +
    CB_WEIGHT * hybrid_df['cb_score_norm'] +
    AFFINITY_WEIGHT * hybrid_df['affinity_score_norm']
)

print("✅ Hybrid scores calculated!")
print(f"Total rows: {len(hybrid_df):,}")
print("\nSample:")
print(hybrid_df.head(10))

✅ Hybrid scores calculated!
Total rows: 55,565

Sample:
  customer_id        product_name  als_score_norm  cb_score_norm  \
0     C_00001    checking_account        0.968013       0.000000   
1     C_00001       fixed_deposit        0.037283       0.000000   
2     C_00001     investment_fund        0.000000       0.742317   
3     C_00001       personal_loan        0.000000       0.485780   
4     C_00001     savings_account        0.978191       0.000000   
5     C_00001  travel_credit_card        0.000000       0.915367   
6     C_00002    checking_account        0.952228       0.000000   
7     C_00002     investment_fund        0.000000       0.721216   
8     C_00002       personal_loan        0.000000       0.356881   
9     C_00002     savings_account        0.055561       0.377179   

              segment  affinity_score  affinity_score_norm  hybrid_score  
0  young_professional           1.000             1.000000      0.587205  
1  young_professional           0.052        

In [8]:
# Load user item matrix to know what's owned
user_item = read_table('GOLD_USER_ITEM_MATRIX')
user_item.columns = user_item.columns.str.lower()

# Create owned products set per customer
owned_lookup = {}
product_columns = ['checking_account', 'savings_account', 'travel_credit_card',
                   'cashback_credit_card', 'personal_loan', 'home_loan',
                   'investment_fund', 'fixed_deposit']

for _, row in user_item.iterrows():
    owned = set()
    for col in product_columns:
        if row.get(col, 0) == 1:
            owned.add(col)
    owned_lookup[row['customer_id']] = owned

# Filter out owned products from hybrid_df
hybrid_df['is_owned'] = hybrid_df.apply(
    lambda row: row['product_name'] in owned_lookup.get(row['customer_id'], set()),
    axis=1
)
hybrid_unowned = hybrid_df[hybrid_df['is_owned'] == False].copy()

# Get top 3 unowned recommendations per customer
final_recommendations = (
    hybrid_unowned
    .sort_values('hybrid_score', ascending=False)
    .groupby('customer_id')
    .head(3)
    .sort_values(['customer_id', 'hybrid_score'], ascending=[True, False])
    .reset_index(drop=True)
)

print("✅ Final recommendations generated!")
print(f"Total: {len(final_recommendations):,}")
print(f"Unique customers: {final_recommendations['customer_id'].nunique():,}")
print("\nSample — C_00001:")
print(final_recommendations[final_recommendations['customer_id'] == 'C_00001']
      [['customer_id', 'product_name', 'hybrid_score']])

✅ Loaded GOLD_USER_ITEM_MATRIX: 10,000 rows
✅ Final recommendations generated!
Total: 29,966
Unique customers: 10,000

Sample — C_00001:
  customer_id        product_name  hybrid_score
0     C_00001  travel_credit_card      0.366980
1     C_00001     investment_fund      0.299010
2     C_00001       personal_loan      0.196395


In [9]:
# Save final recommendations
final_recommendations.to_csv('../data/hybrid_recommendations.csv', index=False)

# Log to MLflow
mlflow.set_tracking_uri('file:///C:/Users/Foram/Projects/Fintech Recommender System/fintech-recommender/mlflow')
mlflow.set_experiment("fintech_recommender_hybrid")

with mlflow.start_run(run_name="Hybrid_v1"):
    mlflow.log_param("als_weight", 0.4)
    mlflow.log_param("cb_weight", 0.4)
    mlflow.log_param("affinity_weight", 0.2)
    mlflow.log_param("data_source", "Snowflake Gold")
    mlflow.log_metric("total_recommendations", len(final_recommendations))
    mlflow.log_metric("unique_customers", final_recommendations['customer_id'].nunique())

print("✅ Hybrid model complete!")
print(f"Saved: hybrid_recommendations.csv")
print(f"Total: {len(final_recommendations):,}")
print(f"Unique customers: {final_recommendations['customer_id'].nunique():,}")

✅ Hybrid model complete!
Saved: hybrid_recommendations.csv
Total: 29,966
Unique customers: 10,000
